# Sugar Sugar study analysis — walkthrough

This notebook is a **thin UI over the library**. Every analysis step calls
`sugar_data_processing` the same way the CLI does. Reports are always written to
`output/reports/` (not a notebook-specific folder).

## Research question

How well can people predict **next-hour glucose** from recent CGM context,
and does accuracy differ by diabetes status, CGM use, experience, or own vs
generic data?

H1–H4 are reported **category first** (the four diabetes × CGM buckets), then
each category is split into **own data** vs **generic data**. H3 and H4 measure
duration / CGM experience in **months** so short experience is not crushed on a year axis.

## Pipeline stages (library modules)

| Step | Stage | Module |
| --- | --- | --- |
| 1 | Gathering | `sugar_data_processing.gathering` |
| 2 | Verification | `sugar_data_processing.verification` |
| 3 | Statistics | `sugar_data_processing.statistics` |
| 4 | Comparison | `sugar_data_processing.comparison` |
| 5 | Output | `sugar_data_processing.output` → `output/reports/` |

**MAE** = mean absolute error (mg/dL). Lower is better.  
**Person-level MAE** = one score per participant (mean of their round MAEs).

Select the `sugar-data-processing (.venv)` kernel, then run cells top to bottom.

## 0. Setup (library session)

`prepare_session` configures the same default CSV and `output/` tree as the CLI.

In [ ]:
from sugar_data_processing.runtime import prepare_session

session = prepare_session(globals())

import polars as pl

from sugar_data_processing import run_analysis
from sugar_data_processing.comparison import benchmark_context
from sugar_data_processing.gathering import (
    build_participant_table,
    load_prediction_statistics,
)
from sugar_data_processing.output import (
    explain_all_hypotheses,
    explain_benchmarks,
    explain_gathering,
    explain_report_written,
    explain_verification,
    show_report_figures,
    write_report,
)
from sugar_data_processing.statistics import hypotheses_table, run_all_hypotheses
from sugar_data_processing.verification import verify_dataset

CSV_PATH = session.csv_path
OUTPUT_DIR = session.output_dir

print("Library session ready.")
print(f"  Repository : {session.repo_root}")
print(f"  Input CSV  : {CSV_PATH}")
print(f"  Output dir : {OUTPUT_DIR}")
print(f"  Reports    : {session.reports_dir}")

## Map of the hypotheses

Catalog comes from `sugar_data_processing.statistics.hypotheses_table()`.

In [ ]:
hypotheses_table()

## 1. Data gathering

Load the statistics export, expand rounds, and build **one row per participant**.

Library calls: `load_prediction_statistics`, `build_participant_table`, `explain_gathering`.

In [ ]:
runs = load_prediction_statistics(CSV_PATH)
participants = build_participant_table(runs)

print(explain_gathering(runs, participants))
participants.select(
    "study_id",
    "diabetic",
    "uses_cgm",
    "mae_primary",
    "mae_generic",
    "mae_own",
    "diabetes_duration_months",
    "cgm_duration_months",
    "n_rounds_generic",
    "n_rounds_own",
    "eligible_primary",
    "eligible_h5",
).head(8)

## 2. Data verification

Schema checks first, then demographic / metric quality flags.

Library calls: `verify_dataset`, `explain_verification`.

In [ ]:
verification = verify_dataset(runs, participants)
print(explain_verification(verification))

## 3. Statistical tests (H1–H5)

Run the study-design battery on person-level MAE. H1–H4 print the four
categories first, then the same test on generic data and on own data.
H3 and H4 use duration / CGM experience in months.

Library calls: `run_all_hypotheses`, `explain_all_hypotheses`.

In [ ]:
suite = run_all_hypotheses(participants)
print(explain_all_hypotheses(suite.to_dict()))
if suite.notes:
    print("Pipeline notes:")
    for note in suite.notes:
        print(f"  - {note}")

## 4. Literature / GlucoBench comparison

Place human MAE next to published 60-minute bands (context for H6, which is deferred).

Library calls: `benchmark_context`, `explain_benchmarks`.

In [ ]:
eligible = participants.filter(pl.col("eligible_primary"))
bench_frame = eligible if eligible.height > 0 else participants
benchmarks = benchmark_context(bench_frame, mae_col="mae_primary")
print(explain_benchmarks(benchmarks))

## 5. Output — always `output/reports/`

Write the human markdown report, figures, and interactive `human_explorer.html` to the **same** folder the CLI uses.

Library calls: `write_report`, `explain_report_written`, `show_report_figures`.

In [ ]:
report_path = write_report(
    runs=runs,
    participants=participants,
    suite=suite,
    benchmarks=benchmarks,
    verification=verification,
    output_dir=OUTPUT_DIR,
    source_csv=CSV_PATH,
)
print(explain_report_written(report_path))
show_report_figures(OUTPUT_DIR)

## 6. One-shot library entry point

Same five stages via `run_analysis` — identical to
`uv run sugar-data-processing analyze --fixture`.

In [ ]:
result = run_analysis(CSV_PATH, OUTPUT_DIR)
print("One-shot pipeline finished.")
print(f"  Report              : {result.report_path}")
print(f"  Verification passed : {result.verification.passed}")
print(f"  Mean person MAE     : {result.benchmarks.human_mean_mae:.2f} mg/dL")
print("\nReadable deliverable: output/reports/human_analysis_report.md")
print("Interactive explorer : output/reports/human_explorer.html")